In [ ]:
# %%
import math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("results/figs", exist_ok=True)
plt.rcParams["figure.figsize"] = (9, 5)
QUICK = False
N_SEEDS = 40 if QUICK else 200

# ---- same world as the notebook (do not change, or results will not match) ----
_layout = np.random.default_rng(7)
N_NODES = 12
XY = _layout.uniform(0, 900, size=(N_NODES, 2))
SPEED = 130.0
T = np.ceil(np.linalg.norm(XY[:, None] - XY[None], axis=2) / SPEED).astype(int)
N_BINS = N_NODES - 1
BASE_RATE = _layout.uniform(20, 48, size=N_BINS)
DAY_MIN, START_HOUR = 720, 8


def demand_curve(h):
    return (0.25 + 1.2 * math.exp(-((h - 12.5) / 1.2) ** 2)
            + 0.6 * math.exp(-((h - 10.0) / 0.6) ** 2)
            + 0.6 * math.exp(-((h - 15.0) / 0.7) ** 2))


CURVE = np.array([demand_curve(START_HOUR + t / 60) for t in range(DAY_MIN)])


def fill_paths(seed, scale=1.0):
    """Fill level (%) of every bin every minute with NO robots: shape (DAY_MIN, N_BINS).
    Uses the same RNG draw order as simulate() so seed k is the same day."""
    rng = np.random.default_rng(seed)
    day_mult = rng.lognormal(0, 0.2, N_BINS)
    noise = np.clip(1 + 0.3 * rng.standard_normal((DAY_MIN, N_BINS)), 0, None)
    start = rng.uniform(0, 50, N_BINS)
    inc = BASE_RATE * day_mult * CURVE[:, None] * scale * noise / 60
    return np.minimum(start + np.cumsum(inc, axis=0), 100.0), day_mult


PATHS = [fill_paths(s)[0] for s in range(N_SEEDS)]
DAYMULT = np.array([fill_paths(s)[1] for s in range(N_SEEDS)])
print(f"{N_SEEDS} simulated days, {N_BINS} bins")

# %% [markdown]
# ## C. Fill forecasting vs baselines  (optimize.py: predictive dispatch)
# Task: predict a bin's fill 30 minutes ahead, then flag "will cross 80% soon".
# Train and test use DIFFERENT simulated days. Baselines first, then a model.
# Caveat to say out loud: the data is simulated, so this shows the method and the
# value of each feature; rerun on real sensor logs before claiming real accuracy.

# %%
H = 30
TRAIN = range(0, N_SEEDS * 2 // 3)
TEST = range(N_SEEDS * 2 // 3, N_SEEDS)


def make_rows(seeds):
    X, y, meta = [], [], []
    for s in seeds:
        P = PATHS[s]
        for t in range(60, DAY_MIN - H, 10):
            cur, slope30, slope60 = P[t], P[t] - P[t - 30], P[t] - P[t - 60]
            ahead = CURVE[t:t + H].mean()                     # known schedule (class breaks, lunch)
            for i in range(N_BINS):
                if cur[i] >= 80:                              # already due; not a forecasting case
                    continue
                X.append([cur[i], slope30[i], slope60[i], ahead, BASE_RATE[i] * ahead,
                          CURVE[t - 30:t].mean() * BASE_RATE[i]])
                y.append(P[t + H][i] - cur[i])
                meta.append((s, t, i, cur[i]))
    return np.array(X), np.array(y), pd.DataFrame(meta, columns=["seed", "t", "bin", "cur"])


Xtr, ytr, mtr = make_rows(TRAIN)
Xte, yte, mte = make_rows(TEST)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9


def ridge_fit(X, y, lam=1.0):
    A = np.c_[np.ones(len(X)), (X - mu) / sd]
    return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ y)


def ridge_pred(w, X):
    return np.c_[np.ones(len(X)), (X - mu) / sd] @ w


w = ridge_fit(Xtr, ytr)
pred = {
    "naive: global mean": np.full(len(yte), ytr.mean()),
    "persistence: repeat last 30 min": Xte[:, 1],
    "base rate x schedule (no live data)": Xte[:, 4] / 60 * H,
    "ridge (slope + schedule)": ridge_pred(w, Xte),
}
res = []
for name, p in pred.items():
    err = yte - p
    cls_true = (mte.cur.values + yte) >= 80
    cls_pred = (mte.cur.values + p) >= 80
    tp = (cls_true & cls_pred).sum()
    res.append({"model": name, "MAE_pct": np.abs(err).mean(), "RMSE_pct": math.sqrt((err ** 2).mean()),
                "R2": 1 - (err ** 2).sum() / ((yte - yte.mean()) ** 2).sum(),
                "alert_precision": tp / max(cls_pred.sum(), 1), "alert_recall": tp / max(cls_true.sum(), 1)})
res = pd.DataFrame(res)
res.to_csv("results/forecast_eval.csv", index=False)
print(f"\nForecast of fill {H} min ahead, tested on {len(set(mte.seed))} unseen days:")
print(res.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].scatter(yte[::20], pred["ridge (slope + schedule)"][::20], s=6, alpha=0.4)
lim = [0, max(yte.max(), 1)]
ax[0].plot(lim, lim, "k--"); ax[0].set(xlabel="actual change in 30 min (%)", ylabel="predicted", title="Ridge: predicted vs actual")
ax[1].bar(range(len(res)), res.MAE_pct, tick_label=["naive", "persist", "rate x sched", "ridge"])
ax[1].set(ylabel="MAE (fill %)", title="Held-out error by model"); plt.tight_layout()
plt.savefig("results/figs/C_forecast.png", dpi=130); plt.show()